<a href="https://colab.research.google.com/github/MehmetErenBekar/Bonus/blob/main/C5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# -*- coding: utf-8 -*-
"""
🚀 CRYPTO BOT C5 - THE REFINER (TZ-FIXED)
====================================================
✅ DÜZELTME: tz-naive ve tz-aware DatetimeIndex join hatası çözüldü (Yahoo Finance)
✅ TRIPLE BARRIER: Stop/Profit/Time-based labeling
✅ SOFT RECENCY WEIGHTING: Son performansa göre ağırlıklandırma
✅ FEATURE GATEKEEPER: Her coin için optimal feature set
"""

# =====================================================
# 1. KULLANICI AYARLARI
# =====================================================
class UserConfig:


    SYMBOLS = ["BTC", "DOGE", "ETH", "SOL", "AVAX"]
    TARGET_BUY_PRICES = {"ETH": 2600, "BTC": 85000, "DOGE": 0.12, "SOL": 0, "AVAX": 0}
    TARGET_SELL_PRICES = {"ETH": 3200, "BTC": 95000, "DOGE": 0.20, "SOL": 160, "AVAX": 0}

    NOTIFY_THRESHOLD = 0.01
    MAX_DATA_LIMIT = 2000
    CRISIS_DATA_LIMIT = 300

    TEST_SIZE = 24
    SEQ_LEN = 12
    OPTUNA_TRIALS = 5
    TIMEFRAME = '15m'
    ACCURACY_THRESHOLD = 51.0

    # Triple Barrier Parametreleri
    PROFIT_TARGET = 0.02
    STOP_LOSS = 0.01
    TIME_BARRIER = 4

    PAPER_TRADING_ENABLED = True
    INITIAL_CAPITAL = 10000
    MAX_RISK_PER_TRADE = 0.02
    MIN_CONFIDENCE_FOR_TRADE = 60
    MIN_KELLY_FOR_TRADE = 3.0

    BRAIN_FILE = "/content/drive/MyDrive/omega_brain.pkl"
    PORTFOLIO_FILE = "/content/drive/MyDrive/crypto_portfolio.json"
    FEATURE_MEMORY_FILE = "/content/drive/MyDrive/feature_memory.json"

# =====================================================
# 2. KURULUM & DRIVE
# =====================================================
import sys, subprocess, warnings, time, smtplib, os, pickle, requests, json
from datetime import datetime
import numpy as np
import pandas as pd
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Drive Bağlandı! Hafıza güvende.")
except:
    print("⚠️ Drive Bağlanamadı! (Lokal mod).")
    UserConfig.BRAIN_FILE = "omega_brain.pkl"
    UserConfig.PORTFOLIO_FILE = "crypto_portfolio.json"
    UserConfig.FEATURE_MEMORY_FILE = "feature_memory.json"

def install_libs():
    print("📦 SİSTEM YÜKLENİYOR... (C5 Version)")
    packages = ['textblob', 'requests', 'pandas', 'numpy', 'scikit-learn', 'ta', 'yfinance', 'xgboost', 'transformers', 'torch', 'optuna']
    for p in packages:
        try: __import__(p)
        except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", p, "-q"])
    print("✅ Sistem Hazır.\n")

install_libs()

import ta, yfinance as yf, torch, optuna, xgboost as xgb
import torch.nn as nn, torch.optim as optim
import xml.etree.ElementTree as ET
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from transformers import pipeline

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# =====================================================
# 3. TRIPLE BARRIER LABELING
# =====================================================
class TripleBarrierLabeler:

    @staticmethod
    def apply_triple_barrier(prices, profit_target=0.02, stop_loss=0.01, time_barrier=4):
        labels = []
        returns = []

        for i in range(len(prices) - time_barrier):
            entry_price = prices[i]
            future_prices = prices[i+1:i+time_barrier+1]

            price_changes = (future_prices - entry_price) / entry_price

            hit_stop = np.any(price_changes <= -stop_loss)
            hit_profit = np.any(price_changes >= profit_target)

            if hit_stop:
                stop_idx = np.argmax(price_changes <= -stop_loss)
                actual_return = price_changes[stop_idx] - 0.001
                labels.append(-1)
                returns.append(actual_return)
            elif hit_profit:
                profit_idx = np.argmax(price_changes >= profit_target)
                actual_return = price_changes[profit_idx] - 0.001
                labels.append(1)
                returns.append(actual_return)
            else:
                actual_return = price_changes[-1] - 0.001
                labels.append(0)
                returns.append(actual_return)

        labels.extend([np.nan] * time_barrier)
        returns.extend([np.nan] * time_barrier)

        return np.array(labels), np.array(returns)

# =====================================================
# 4. SOFT RECENCY WEIGHTING
# =====================================================
class RecencyWeighter:

    def __init__(self, window=12):
        self.window = window
        self.performance_history = {
            'XGB': [],
            'RF': [],
            'LSTM': []
        }

    def update_performance(self, model_name, accuracy):
        if model_name in self.performance_history:
            self.performance_history[model_name].append(accuracy)
            if len(self.performance_history[model_name]) > self.window:
                self.performance_history[model_name].pop(0)

    def get_weights(self):
        weights = {}

        for model_name, history in self.performance_history.items():
            if len(history) > 0:
                avg_perf = np.mean(history)
                weights[model_name] = avg_perf
            else:
                weights[model_name] = 50.0

        total = sum(weights.values())
        if total > 0:
            for k in weights:
                weights[k] = weights[k] / total
                weights[k] = max(0.20, min(0.50, weights[k]))
        else:
            weights = {k: 0.33 for k in weights}

        return weights

# =====================================================
# 5. PAPER TRADING PORTFOLIO
# =====================================================
class PaperPortfolio:
    def __init__(self, filepath):
        self.filepath = filepath
        self.load_or_create()

    def load_or_create(self):
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, 'r') as f:
                    data = json.load(f)
                    self.balance = data['balance']
                    self.positions = data['positions']
                    self.trade_history = data['trade_history']
                    self.start_date = data.get('start_date', datetime.now().isoformat())
                    print(f"   💰 PORTFÖY YÜKLENDI: ${self.balance:,.2f} bakiye, {len(self.positions)} pozisyon")
            except:
                self._create_new()
        else:
            self._create_new()

    def _create_new(self):
        self.balance = UserConfig.INITIAL_CAPITAL
        self.positions = {}
        self.trade_history = []
        self.start_date = datetime.now().isoformat()
        self.save()
        print(f"   🆕 YENİ PORTFÖY OLUŞTURULDU: ${self.balance:,.2f}")

    def save(self):
        try:
            data = {
                'balance': self.balance,
                'positions': self.positions,
                'trade_history': self.trade_history,
                'start_date': self.start_date,
                'last_update': datetime.now().isoformat()
            }
            with open(self.filepath, 'w') as f:
                json.dump(data, f, indent=2)
        except: pass

    def get_total_value(self, current_prices):
        total = self.balance
        for symbol, pos in self.positions.items():
            if symbol in current_prices:
                total += pos['size'] * current_prices[symbol]
        return total

    def calculate_position_size(self, current_price, kelly_pct, confidence):
        safe_kelly = (kelly_pct / 100) * 0.25
        confidence_mult = min(confidence / 100, 1.0)
        position_value = self.balance * safe_kelly * confidence_mult
        max_position = self.balance * UserConfig.MAX_RISK_PER_TRADE
        position_value = min(position_value, max_position)
        return position_value / current_price

    def open_position(self, symbol, price, size, reason):
        cost = size * price * (1 + 0.001)
        if cost > self.balance:
            return False

        self.balance -= cost
        self.positions[symbol] = {
            'size': size,
            'entry_price': price,
            'entry_date': datetime.now().isoformat(),
            'reason': reason
        }
        self.trade_history.append({
            'type': 'BUY',
            'symbol': symbol,
            'price': price,
            'size': size,
            'date': datetime.now().isoformat()
        })
        self.save()
        return True

    def close_position(self, symbol, price, reason):
        if symbol not in self.positions:
            return False, 0

        pos = self.positions[symbol]
        revenue = pos['size'] * price * (1 - 0.001)
        self.balance += revenue

        pnl = revenue - (pos['size'] * pos['entry_price'])
        pnl_pct = (pnl / (pos['size'] * pos['entry_price'])) * 100

        self.trade_history.append({
            'type': 'SELL',
            'symbol': symbol,
            'price': price,
            'pnl': pnl,
            'pnl_pct': pnl_pct,
            'date': datetime.now().isoformat()
        })

        del self.positions[symbol]
        self.save()
        return True, pnl_pct

# =====================================================
# 6. İSTİHBARAT
# =====================================================
class MoneyFlowIntelligence:
    @staticmethod
    def analyze_flow(df):
        try:
            mfi = ta.volume.money_flow_index(df['high'], df['low'], df['close'], df['volume'], window=14).iloc[-1]
            vwap = (df['volume'] * (df['high'] + df['low'] + df['close']) / 3).rolling(24).sum() / df['volume'].rolling(24).sum()
            current_price = df['close'].iloc[-1]
            vwap_bullish = current_price > vwap.iloc[-1]
            flow_score = (mfi / 20.0) + (1.0 if vwap_bullish else 0)
            return flow_score, mfi, vwap_bullish
        except: return 0, 50, False

class FinBERTEngine:
    _sentiment_pipeline = None
    @classmethod
    def get_sentiment(cls, text):
        if cls._sentiment_pipeline is None:
            cls._sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert")
        try:
            r = cls._sentiment_pipeline(text[:512])[0]
            return 50 + (r['score']*50) if r['label'] == 'positive' else (50 - (r['score']*50) if r['label'] == 'negative' else 50)
        except: return 50

class NewsIntelligence:
    @staticmethod
    def analyze(symbol):
        try:
            r = requests.get(f"https://news.google.com/rss/search?q={symbol}%20crypto&hl=en-US&gl=US&ceid=US:en", timeout=5)
            root = ET.fromstring(r.content)
            headlines = [i.find('title').text for i in root.findall('./channel/item')[:5]]
            return (FinBERTEngine.get_sentiment(" ".join(headlines)), headlines[0]) if headlines else (50, "Haber Yok")
        except: return 50, "Veri Hatası"

class MarketPsychology:
    @staticmethod
    def get_fear_and_greed():
        try:
            r = requests.get("https://api.alternative.me/fng/", timeout=5).json()
            return int(r['data'][0]['value']), r['data'][0]['value_classification']
        except: return 50, "Nötr"

    @staticmethod
    def get_fear_history():
        try:
            r = requests.get("https://api.alternative.me/fng/?limit=300&format=json", timeout=10).json()
            df = pd.DataFrame(r['data'])
            df['value'] = df['value'].astype(int)
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
            return df.set_index('timestamp').resample('H').ffill()[['value']].rename(columns={'value': 'Fear_Index'})
        except: return None

# =====================================================
# 7. DATA PROVIDER (🔧 YAHOO FINANCE & TZ FIX)
# =====================================================
class DataProvider:
    @staticmethod
    def get_crypto_data(symbol, limit):
        try:
            yahoo_map = {
                "BTC": "BTC-USD",
                "ETH": "ETH-USD",
                "DOGE": "DOGE-USD",
                "SOL": "SOL-USD",
                "AVAX": "AVAX-USD"
            }

            if symbol not in yahoo_map:
                return None

            ticker = yf.Ticker(yahoo_map[symbol])
            period = "60d" if UserConfig.TIMEFRAME == '15m' else "90d"
            interval = "15m" if UserConfig.TIMEFRAME == '15m' else "1h"

            df = ticker.history(period=period, interval=interval)

            if df.empty or len(df) < 50:
                return None

            df = df.rename(columns={
                'Open': 'open',
                'High': 'high',
                'Low': 'low',
                'Close': 'close',
                'Volume': 'volume'
            })

            df = df[['open', 'high', 'low', 'close', 'volume']]
            df.index.name = 'timestamp'

            # ✅ KRİTİK TZ DÜZELTMESİ (Join Hatasını Çözer)
            if df.index.tz is not None:
                df.index = df.index.tz_localize(None)

            if len(df) > limit:
                df = df.tail(limit)

            return df.astype(float)

        except:
            return None

    @staticmethod
    def get_macro_data(limit):
        try:
            tickers = ["DX-Y.NYB", "^GSPC", "^TNX", "GC=F"]
            macro = yf.download(tickers, period="5d", interval="15m", progress=False)['Close']
            if macro.empty: return None
            if macro.index.tz is not None: macro.index = macro.index.tz_localize(None)
            macro = macro.ffill().fillna(method='bfill')
            if isinstance(macro.columns, pd.MultiIndex): macro.columns = [col[0] for col in macro.columns]
            rename_map = {"DX-Y.NYB": "DXY", "^GSPC": "SPX", "^TNX": "US10Y", "GC=F": "GOLD"}
            macro = macro.rename(columns={k:v for k,v in rename_map.items() if k in macro.columns})
            return macro
        except: return None

class CurrencyEngine:
    @staticmethod
    def get_usd_try_rate():
        try: return float(yf.download("TRY=X", period="1d", progress=False)['Close'].iloc[-1])
        except: return 34.0

# =====================================================
# 8. RL BRAIN
# =====================================================
class RLBrain:
    def __init__(self):
        self.actions = ['DEFENSIVE', 'NEUTRAL', 'AGGRESSIVE']
        self.q_table = pd.DataFrame(columns=self.actions, dtype=np.float64)
        if os.path.exists(UserConfig.BRAIN_FILE):
            try:
                with open(UserConfig.BRAIN_FILE, 'rb') as f: self.q_table = pickle.load(f)
                print(f"   💾 HAFIZA OKUNDU: {len(self.q_table)} durum.")
            except: print("   ⚠️ Hafıza yenilendi.")

    def choose_action(self, state):
        if state not in self.q_table.index:
            self.q_table = pd.concat([self.q_table, pd.DataFrame([[0]*3], index=[state], columns=self.actions)])
        return self.q_table.loc[state].idxmax() if np.random.uniform() < 0.9 else np.random.choice(self.actions)

    def learn(self, s, a, r, s_):
        if s not in self.q_table.index: return
        q_cur = self.q_table.loc[s, a]
        q_target = r + 0.9 * self.q_table.loc[s_, :].max() if s_ != 'TERMINAL' and s_ in self.q_table.index else r
        self.q_table.loc[s, a] += 0.1 * (q_target - q_cur)
        try:
            with open(UserConfig.BRAIN_FILE, 'wb') as f: pickle.dump(self.q_table, f)
        except: pass

# =====================================================
# 9. FEATURE GATEKEEPER
# =====================================================
class FeatureGatekeeper:
    def __init__(self):
        self.memory_file = UserConfig.FEATURE_MEMORY_FILE
        self.feature_memory = self.load_memory()

    def load_memory(self):
        if os.path.exists(self.memory_file):
            try:
                with open(self.memory_file, 'r') as f:
                    memory = json.load(f)
                    print(f"   🧠 FEATURE HAFIZASI YÜKLENDI: {len(memory)} coin kaydı")
                    return memory
            except:
                return {}
        return {}

    def save_memory(self):
        try:
            with open(self.memory_file, 'w') as f:
                json.dump(self.feature_memory, f, indent=2)
        except: pass

    @staticmethod
    def calculate_mutual_information(X, y, feature_names):
        try:
            mi_scores = mutual_info_regression(X, y, random_state=42)
            mi_dict = {name: score for name, score in zip(feature_names, mi_scores)}
            return mi_dict
        except:
            return {name: 0.5 for name in feature_names}

    @staticmethod
    def remove_redundant_features(X, feature_names, threshold=0.95):
        correlation_matrix = np.corrcoef(X.T)
        to_remove = set()
        for i in range(len(feature_names)):
            for j in range(i+1, len(feature_names)):
                if abs(correlation_matrix[i, j]) > threshold:
                    to_remove.add(feature_names[j])
        clean_features = [f for f in feature_names if f not in to_remove]
        return clean_features, list(to_remove)

    def select_best_features(self, symbol, df_baseline, df_advanced, X_baseline, X_advanced, y, baseline_features, advanced_features):
        print(f"      🔬 FEATURE GATEKEEPER: {symbol} analiz ediliyor...")

        mi_scores = self.calculate_mutual_information(X_advanced, y, advanced_features)
        low_mi_features = [f for f, score in mi_scores.items() if score < 0.01]
        if low_mi_features:
            print(f"         ⚠️ Düşük MI: {', '.join(low_mi_features)}")

        clean_advanced, removed = self.remove_redundant_features(X_advanced, advanced_features, threshold=0.95)
        if removed:
            print(f"         🧹 Gereksiz: {', '.join(removed)}")

        try:
            m_baseline = xgb.XGBRegressor(n_estimators=50, max_depth=5).fit(X_baseline[:-24], y[:-24])
            acc_baseline = np.mean(np.sign(m_baseline.predict(X_baseline[-24:])) == np.sign(y[-24:])) * 100

            m_advanced = xgb.XGBRegressor(n_estimators=50, max_depth=5).fit(X_advanced[:-24], y[:-24])
            acc_advanced = np.mean(np.sign(m_advanced.predict(X_advanced[-24:])) == np.sign(y[-24:])) * 100

            winner = "ADVANCED" if acc_advanced > acc_baseline + 2.0 else "BASELINE"

            print(f"         🏆 TURNUVA: Baseline %{acc_baseline:.1f} vs Advanced %{acc_advanced:.1f}")
            print(f"         ✅ KAZANAN: {winner} ({acc_advanced - acc_baseline:+.1f}% fark)")

            self.feature_memory[symbol] = {
                'winner': winner,
                'baseline_acc': acc_baseline,
                'advanced_acc': acc_advanced,
                'timestamp': datetime.now().isoformat()
            }
            self.save_memory()

            if winner == "ADVANCED":
                return advanced_features, "C5-ADVANCED", acc_advanced
            else:
                return baseline_features, "C5-BASELINE", acc_baseline

        except Exception as e:
            print(f"         ⚠️ Tournament hatası: {e}, BASELINE kullanılacak")
            return baseline_features, "C5-BASELINE-SAFE", 50.0

# =====================================================
# 10. MODELS
# =====================================================
class LSTMNet(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(LSTMNet, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class NeuralTrainer:
    @staticmethod
    def create_sequences(X, y, seq_len):
        Xs, ys = [], []
        for i in range(len(X) - seq_len - 1):
            Xs.append(X[i:(i+seq_len)])
            ys.append(y[i+seq_len])
        return np.array(Xs), np.array(ys)

    @staticmethod
    def train_lstm(X, y, params):
        model = LSTMNet(X.shape[2], params['hidden_dim']).to(device)
        optimizer = optim.Adam(model.parameters(), lr=params['lr'])
        X_t, y_t = torch.FloatTensor(X).to(device), torch.FloatTensor(y).view(-1,1).to(device)
        model.train()
        for _ in range(30):
            optimizer.zero_grad()
            nn.MSELoss()(model(X_t), y_t).backward()
            optimizer.step()
        return model

class OptunaBrain:
    @staticmethod
    def objective_xgb(trial, X, y, sw):
        p = {'n_estimators': trial.suggest_int('n_estimators', 50, 300), 'max_depth': trial.suggest_int('max_depth', 3, 10), 'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3)}
        m = xgb.XGBRegressor(**p).fit(X[:-24], y[:-24], sample_weight=sw[:-24])
        return np.mean(np.sign(m.predict(X[-24:])) == np.sign(y[-24:]))

    @staticmethod
    def objective_rf(trial, X, y, sw):
        p = {'n_estimators': trial.suggest_int('n_estimators', 50, 200), 'max_depth': trial.suggest_int('max_depth', 5, 20), 'min_samples_split': trial.suggest_int('min_samples_split', 2, 10)}
        m = RandomForestRegressor(**p).fit(X[:-24], y[:-24], sample_weight=sw[:-24])
        return np.mean(np.sign(m.predict(X[-24:])) == np.sign(y[-24:]))

    @staticmethod
    def objective_lstm(trial, X, y, dim):
        p = {'hidden_dim': trial.suggest_categorical('hidden_dim', [32, 64]), 'lr': trial.suggest_float('lr', 1e-4, 1e-2)}
        m = NeuralTrainer.train_lstm(X[:-24], y[:-24], p)
        m.eval()
        with torch.no_grad(): preds = m(torch.FloatTensor(X[-24:]).to(device)).cpu().numpy().flatten()
        return np.mean(np.sign(preds) == np.sign(y[-24:]))

# =====================================================
# 11. OMEGA ENGINE C5
# =====================================================
class LimitEngine:
    @staticmethod
    def calculate_levels(df, risk_mode):
        last = df.iloc[-1]
        atr, close = last['ATR'], last['close']
        bb_low = close - last['Dist_Low']

        vol_z = (last['Volatility'] - df['Volatility'].rolling(24).mean().iloc[-1]) / (df['Volatility'].rolling(24).std().iloc[-1] + 1e-9)
        vol_mult = 1.8 if vol_z > 2.0 else (1.4 if vol_z > 1.0 else (0.8 if vol_z < -1.0 else 1.0))
        base = 3.0 if risk_mode == 'DEFENSIVE' else (1.5 if risk_mode == 'AGGRESSIVE' else 2.0)
        mult = base * vol_mult

        pusu = bb_low + (atr * 0.1)
        return pusu, pusu - (atr * mult), pusu + (atr * mult * 2), mult

class OmegaEngine:
    @staticmethod
    def calculate_smart_weights(df):
        weights = np.ones(len(df))
        if 'Fear_Index' in df.columns:
            weights = np.where(df['Fear_Index'] < 25, weights * 2.0, weights)
        if 'Volatility' in df.columns:
            weights = np.where(df['Volatility'] > df['Volatility'].mean(), weights * 1.5, weights)
        return weights

    @staticmethod
    def run_ensemble_c5(symbol, fear_val, gatekeeper, recency_weighter):
        data_limit = UserConfig.CRISIS_DATA_LIMIT if fear_val < 25 else UserConfig.MAX_DATA_LIMIT

        df = DataProvider.get_crypto_data(symbol, data_limit)
        raw_macro = DataProvider.get_macro_data(data_limit)
        raw_fear = MarketPsychology.get_fear_history()

        if df is None: return None

        if raw_macro is not None:
            df = df.join(raw_macro, how='left').ffill().bfill()
        if raw_fear is not None:
            df = df.join(raw_fear, how='left').ffill().bfill()

        df['log_ret'] = np.log(df['close']/df['close'].shift(1))
        df['ATR'] = ta.volatility.average_true_range(df['high'], df['low'], df['close'])
        df['Volatility'] = df['log_ret'].rolling(24).std()
        df['Dist_Low'] = df['close'] - ta.volatility.BollingerBands(df['close']).bollinger_lband()
        df['RSI'] = ta.momentum.rsi(df['close'])

        if 'GOLD' in df.columns and df['GOLD'].notna().any():
            df['BTC_GOLD_Ratio'] = df['close'] / (df['GOLD'] + 1e-9)
        if 'DXY' in df.columns and df['DXY'].notna().any():
            df['BTC_DXY_Ratio'] = df['close'] / (df['DXY'] + 1e-9)
        df['ATR_Percentile'] = df['ATR'].rolling(window=168, min_periods=24).apply(
            lambda x: (x.iloc[-1] > x).sum() / len(x) * 100 if len(x) > 0 else 50
        )
        price_momentum = df['close'].pct_change(5)
        rsi_momentum = df['RSI'].diff(5)
        df['Weak_Momentum'] = ((price_momentum > 0.01) & (rsi_momentum < -2)).astype(int)

        df.dropna(subset=['ATR', 'RSI', 'Volatility', 'log_ret'], inplace=True)
        if len(df) < 50: return None

        tb_labels, tb_returns = TripleBarrierLabeler.apply_triple_barrier(
            df['close'].values,
            profit_target=UserConfig.PROFIT_TARGET,
            stop_loss=UserConfig.STOP_LOSS,
            time_barrier=UserConfig.TIME_BARRIER
        )

        df['TB_Label'] = tb_labels
        df['TB_Return'] = tb_returns
        df = df[df['TB_Label'].notna()]

        if len(df) < 50: return None

        baseline_features = ['close', 'RSI', 'ATR', 'Volatility', 'Dist_Low']
        if 'Fear_Index' in df.columns: baseline_features.append('Fear_Index')
        if 'DXY' in df.columns: baseline_features.extend(['DXY', 'GOLD', 'US10Y'])

        advanced_features = baseline_features.copy()
        if 'BTC_GOLD_Ratio' in df.columns: advanced_features.append('BTC_GOLD_Ratio')
        if 'BTC_DXY_Ratio' in df.columns: advanced_features.append('BTC_DXY_Ratio')
        if 'ATR_Percentile' in df.columns: advanced_features.append('ATR_Percentile')
        if 'Weak_Momentum' in df.columns: advanced_features.append('Weak_Momentum')

        X_baseline = StandardScaler().fit_transform(df[baseline_features].values)
        X_advanced = StandardScaler().fit_transform(df[advanced_features].values)
        y = df['TB_Label'].values

        selected_features, config_name, gatekeeper_acc = gatekeeper.select_best_features(
            symbol, df, df, X_baseline, X_advanced, y, baseline_features, advanced_features
        )

        X_selected = StandardScaler().fit_transform(df[selected_features].values)
        sw = OmegaEngine.calculate_smart_weights(df)

        s_xgb = optuna.create_study(direction='maximize')
        s_xgb.optimize(lambda t: OptunaBrain.objective_xgb(t, X_selected, y, sw), n_trials=3)
        m_xgb = xgb.XGBRegressor(**s_xgb.best_params).fit(X_selected[:-24], y[:-24], sample_weight=sw[:-24])
        a_xgb = s_xgb.best_value * 100

        s_rf = optuna.create_study(direction='maximize')
        s_rf.optimize(lambda t: OptunaBrain.objective_rf(t, X_selected, y, sw), n_trials=3)
        m_rf = RandomForestRegressor(**s_rf.best_params).fit(X_selected[:-24], y[:-24], sample_weight=sw[:-24])
        a_rf = s_rf.best_value * 100

        X_seq, y_seq = NeuralTrainer.create_sequences(X_selected, y, 12)
        s_lstm = optuna.create_study(direction='maximize')
        s_lstm.optimize(lambda t: OptunaBrain.objective_lstm(t, X_seq, y_seq, X_selected.shape[1]), n_trials=3)
        m_lstm = NeuralTrainer.train_lstm(X_seq[:-24], y_seq[:-24], s_lstm.best_params)
        a_lstm = s_lstm.best_value * 100

        recency_weighter.update_performance('XGB', a_xgb)
        recency_weighter.update_performance('RF', a_rf)
        recency_weighter.update_performance('LSTM', a_lstm)

        model_weights = recency_weighter.get_weights()

        p_xgb = m_xgb.predict(X_selected[-1:])[0]
        p_rf = m_rf.predict(X_selected[-1:])[0]
        with torch.no_grad():
            p_lstm = m_lstm(torch.FloatTensor(X_seq[-1].reshape(1,12,-1)).to(device)).item()

        final_pred = (p_xgb * model_weights['XGB'] +
                      p_rf * model_weights['RF'] +
                      p_lstm * model_weights['LSTM'])

        final_acc = (a_xgb * model_weights['XGB'] +
                     a_rf * model_weights['RF'] +
                     a_lstm * model_weights['LSTM'])

        bt_preds = m_xgb.predict(X_selected[-24:])
        bt_acc = (np.mean(np.sign(bt_preds) == np.sign(y[-24:])) * 100) - 50

        test_returns = df['TB_Return'].iloc[-24:].values
        kelly_pct = 0.0
        if len(test_returns) >= 5:
            wins = [r for r in test_returns if r > 0]
            losses = [r for r in test_returns if r < 0]
            if wins and losses:
                win_rate = len(wins) / len(test_returns)
                avg_win = np.mean(wins)
                avg_loss = abs(np.mean(losses))
                if avg_win > 0:
                    kelly_raw = (win_rate * avg_win - (1-win_rate) * avg_loss) / avg_win
                    kelly_pct = max(0, min(kelly_raw * 25, 25))
            elif wins: kelly_pct = 5.0

        return final_pred, gatekeeper_acc, df, bt_acc, config_name, kelly_pct, len(selected_features), model_weights

# =====================================================
# 12. MAİL
# =====================================================
class StateManager:
    last_sent_levels = {}
    @staticmethod
    def should_send_mail(symbol, current_buy, current_sell, decision):
        last = StateManager.last_sent_levels.get(symbol)
        if last is None: return True
        if last['decision'] != decision and "GÜÇLÜ" in decision: return True
        old_buy = last['buy']
        if abs(current_buy - old_buy) / old_buy > UserConfig.NOTIFY_THRESHOLD: return True
        return False
    @staticmethod
    def update_state(symbol, buy, sell, decision):
        StateManager.last_sent_levels[symbol] = {'buy': buy, 'sell': sell, 'decision': decision}

def send_strategy_alert_tl(symbol, decision, price_usd, price_try, pusu_usd, pusu_try, stop_usd, stop_try, target_usd, target_try, acc, change_pct, headline, config, flow, mfi, multi, mode_str, pusu_pct, target_pct, stop_pct, rl_action, bt_res, kelly, short_signal):
    try:
        subject = f"🇹🇷 C5 REFINER: {symbol} - {decision}"
        short_msg = "⚠️ SHORT SİNYALİ!" if short_signal else "Trend: Nötr/Long"

        body = f"""
        🤖 CRYPTO BOT C5 - THE REFINER
        ================================
        Zaman: 15m | Mod: {mode_str} | {short_msg}
        💰 {symbol}: ${price_usd:.2f} (Approx. ₺{price_try:,.2f})

        📊 ANALİZ RAPORU
        • Model: {config} | Güven: %{acc:.0f}
        • 24s Backtest: %{bt_res:+.2f} | Kelly: %{kelly:.1f}
        • Ajan Kararı: {rl_action} (Stop Çarpanı: {multi:.1f}x)

        📉 SENARYO A (ALIM)
        👉 LİMİT EMRİ: ₺{pusu_try:,.2f} (Şu anki fiyattan %{pusu_pct:.2f} aşağıda)

        📈 SENARYO B (SATIŞ/KORUMA)
        🎯 HEDEF: ₺{target_try:,.2f} (Kâr: +%{target_pct:.2f})
        ⛔ STOP: ₺{stop_try:,.2f} (Zarar: -%{stop_pct:.2f})

        🧠 İSTİHBARAT
        • Money Flow Index (MFI): {mfi:.0f}/100
        • Para Akış Gücü: {flow:.1f} (5 üzerinden)

        📰 SON HABER
        {headline}
        """
        msg = MIMEMultipart()
        msg['From'] = UserConfig.EMAIL_SENDER
        msg['To'] = UserConfig.EMAIL_RECEIVER
        msg['Subject'] = subject
        msg.attach(MIMEText(body, 'plain'))
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
            server.login(UserConfig.EMAIL_SENDER, UserConfig.EMAIL_PASSWORD)
            server.send_message(msg)
    except Exception as e: pass

# =====================================================
# 13. MAIN - C5 REFINER
# =====================================================
def main():
    print("=" * 100)
    print("🚀 CRYPTO BOT C5 - THE REFINER (TZ-FIXED)")
    print("=" * 100)
    print("✨ Triple Barrier Labeling + Soft Recency Weighting")
    print("🔧 Yahoo Finance TZ Fix Aktif")
    print("=" * 100)

    portfolio = None
    if UserConfig.PAPER_TRADING_ENABLED:
        portfolio = PaperPortfolio(UserConfig.PORTFOLIO_FILE)
        print("✅ Paper Trading Aktif (C5 ile)")

    agent = RLBrain()
    gatekeeper = FeatureGatekeeper()
    recency_weighter = RecencyWeighter(window=12)

    try:
        usd_try = CurrencyEngine.get_usd_try_rate()
        f_val, f_class = MarketPsychology.get_fear_and_greed()
        print(f"🌍 Piyasa: {f_class} ({f_val}/100) | 💲 USD/TRY: {usd_try:.2f}\n")
    except: usd_try = 34.0; f_val = 50

    while True:
        print(f"\n🕒 TARAMA: {time.strftime('%H:%M:%S')}")
        print("=" * 100)

        current_prices = {}
        scan_stats = []

        for symbol in UserConfig.SYMBOLS:
            try:
                print(f"\n🔬 {symbol} ANALİZ EDİLİYOR...")
                print("-" * 100)

                res = OmegaEngine.run_ensemble_c5(symbol, f_val, gatekeeper, recency_weighter)
                if not res: continue

                pred, acc, df, bt, cfg, kelly, feat_count, model_weights = res

                scan_stats.append((symbol, acc))

                price = df.iloc[-1]['close']
                current_prices[symbol] = price

                state = f"{f_val}_{'H' if df.iloc[-1]['Volatility'] > 0.02 else 'L'}"
                action = agent.choose_action(state)
                pusu, stop, tp, mult = LimitEngine.calculate_levels(df, action)
                flow, mfi, vwap_up = MoneyFlowIntelligence.analyze_flow(df)
                sent, head = NewsIntelligence.analyze(symbol)

                change = pred * 2.0
                short = mfi > 80 and not vwap_up
                dec = "SHORT/SAT 🔻" if short else ("GÜÇLÜ AL 🚀" if change > 0.5 and sent > 55 and flow > 3 else ("SAT 🔻" if change < -0.5 else "BEKLE"))

                mode_label = "⚠️ KRİZ MODU" if f_val < 25 else "✅ NORMAL MOD"
                if action == 'DEFENSIVE': mode_label += " (Savunma)"

                if portfolio:
                    if symbol in portfolio.positions:
                        if short or (change < -0.5 and acc < 65):
                            success, pnl_pct = portfolio.close_position(symbol, price, "C5: Triple Barrier stop")
                            if success:
                                print(f"   ⚡ C5 POZİSYON KAPATILDI: {pnl_pct:+.2f}%")

                    elif symbol not in portfolio.positions:
                        if (change > 0.5 and acc >= UserConfig.MIN_CONFIDENCE_FOR_TRADE and
                            kelly >= UserConfig.MIN_KELLY_FOR_TRADE and sent > 55 and flow > 3.5 and not short):

                            size = portfolio.calculate_position_size(price, kelly, acc)
                            if size * price > 100:
                                success = portfolio.open_position(symbol, price, size, f"C5: Triple Barrier long (ACC:{acc:.0f}%)")
                                if success:
                                    print(f"   ⚡ C5 POZİSYON AÇILDI: {size:.6f} {symbol}")

                print(f"\n📊 {symbol} RAPOR:")
                print(f"   💰 Fiyat: ${price:.2f}")
                print(f"   🏆 Model: {cfg} | Güven: %{acc:.1f} | Feature Count: {feat_count}")
                print(f"   ⚖️ Model Ağırlıkları: XGB:{model_weights['XGB']:.0%} RF:{model_weights['RF']:.0%} LSTM:{model_weights['LSTM']:.0%}")
                print(f"   📋 Seçilen Feature'lar: {', '.join(selected_features[:5])}..." if len(selected_features) > 5 else f"   📋 Feature'lar: {', '.join(selected_features)}")
                print(f"   🎯 Karar: {dec} | {mode_label}")
                print(f"   📊 Triple Barrier Signal: {pred:.2f} | Backtest: %{bt:+.1f} | Kelly: %{kelly:.1f}")
                print(f"   📉 SENARYO A: Pusu ₺{pusu*usd_try:,.2f} (İndirim: %{((price-pusu)/price)*100:.2f})")
                print(f"   📈 SENARYO B: Hedef ₺{tp*usd_try:,.2f} (+%{((tp-pusu)/pusu)*100:.2f}) | Stop ₺{stop*usd_try:,.2f} (-%{((pusu-stop)/pusu)*100:.2f})")
                print(f"   🧠 İSTİHBARAT: MFI:{mfi:.0f}/100 | Güç:{flow:.1f} | FinBERT:{sent:.0f}")
                print(f"   📰 Haber: {head[:60]}...")
                if short: print("   ⚠️ UYARI: Olası Düşüş (Short Sinyali)")

                if portfolio and symbol in portfolio.positions:
                    pos = portfolio.positions[symbol]
                    unrealized_pnl = ((price - pos['entry_price']) / pos['entry_price']) * 100
                    print(f"   💼 SANAL POZİSYON: {pos['size']:.6f} @ ${pos['entry_price']:.2f} | P&L: {unrealized_pnl:+.2f}%")

                print("-" * 100)

                agent.learn(state, action, 1 if bt > 0 else -1, "TERMINAL")

                if StateManager.should_send_mail(symbol, pusu, tp, dec):
                    if "GÜÇLÜ" in dec or dec == "BEKLE" or "SHORT" in dec:
                        send_strategy_alert_tl(
                            symbol, dec, price, price*usd_try,
                            pusu, pusu*usd_try, stop, stop*usd_try,
                            tp, tp*usd_try, acc, change, head, cfg,
                            flow, mfi, mult, mode_label,
                            ((price-pusu)/price)*100,
                            ((tp-pusu)/pusu)*100,
                            ((pusu-stop)/pusu)*100,
                            action, bt, kelly, short
                        )
                        StateManager.update_state(symbol, pusu, tp, dec)

            except Exception as e:
                print(f"⚠️ Hata ({symbol}): {str(e)[:50]}")
                continue

        # Liyakat Sıralaması
        if scan_stats:
            scan_stats.sort(key=lambda x: x[1], reverse=True)
            summary_line = " | ".join([f"{s}: %{a:.1f}" for s, a in scan_stats])
            print(f"\n🎯 LİYAKAT SIRALAMASI: {summary_line}")

        print("\n" + "=" * 100)
        print("📊 ÖZET")
        print("=" * 100)

        print("🧠 FEATURE SEÇİMLERİ:")
        for symbol, info in gatekeeper.feature_memory.items():
            winner = info['winner']
            diff = info['advanced_acc'] - info['baseline_acc']
            emoji = "🏆" if winner == "ADVANCED" else "🥈"
            print(f"   {emoji} {symbol}: {winner} (Fark: {diff:+.1f}%)")

        if portfolio:
            total_value = portfolio.get_total_value(current_prices)
            roi = ((total_value - UserConfig.INITIAL_CAPITAL) / UserConfig.INITIAL_CAPITAL) * 100

            print(f"\n💰 SANAL PORTFÖY (C5 ile):")
            print(f"   Değer: ${total_value:,.2f} (ROI: {roi:+.2f}%)")
            print(f"   Nakit: ${portfolio.balance:,.2f} | Pozisyon: {len(portfolio.positions)}")

        print("\n💤 Sistem Dinleniyor... (15 Dakika sonra yeni tarama)")
        print("=" * 100)
        time.sleep(900)

if __name__ == "__main__":
    main()

Mounted at /content/drive
✅ Drive Bağlandı! Hafıza güvende.
📦 SİSTEM YÜKLENİYOR... (C5 Version)
✅ Sistem Hazır.

🚀 CRYPTO BOT C5 - THE REFINER (TZ-FIXED)
✨ Triple Barrier Labeling + Soft Recency Weighting
🔧 Yahoo Finance TZ Fix Aktif
   💰 PORTFÖY YÜKLENDI: $10,000.00 bakiye, 0 pozisyon
✅ Paper Trading Aktif (C5 ile)
   💾 HAFIZA OKUNDU: 12 durum.
   🧠 FEATURE HAFIZASI YÜKLENDI: 5 coin kaydı
🌍 Piyasa: Extreme Fear (18/100) | 💲 USD/TRY: 44.11


🕒 TARAMA: 16:06:46

🔬 BTC ANALİZ EDİLİYOR...
----------------------------------------------------------------------------------------------------
      🔬 FEATURE GATEKEEPER: BTC analiz ediliyor...
         ⚠️ Düşük MI: ATR, Volatility, BTC_DXY_Ratio, ATR_Percentile, Weak_Momentum
         🧹 Gereksiz: BTC_DXY_Ratio
         🏆 TURNUVA: Baseline %8.3 vs Advanced %0.0
         ✅ KAZANAN: BASELINE (-8.3% fark)


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


📊 BTC RAPOR:
   💰 Fiyat: $69652.00
   🏆 Model: C5-BASELINE | Güven: %8.3 | Feature Count: 9
   ⚖️ Model Ağırlıkları: XGB:20% RF:50% LSTM:20%
⚠️ Hata (BTC): name 'selected_features' is not defined

🔬 DOGE ANALİZ EDİLİYOR...
----------------------------------------------------------------------------------------------------
      🔬 FEATURE GATEKEEPER: DOGE analiz ediliyor...
         ⚠️ Düşük MI: Weak_Momentum
         🧹 Gereksiz: BTC_DXY_Ratio, Volatility, BTC_GOLD_Ratio
         🏆 TURNUVA: Baseline %12.5 vs Advanced %16.7
         ✅ KAZANAN: ADVANCED (+4.2% fark)

📊 DOGE RAPOR:
   💰 Fiyat: $0.09
   🏆 Model: C5-ADVANCED | Güven: %16.7 | Feature Count: 13
   ⚖️ Model Ağırlıkları: XGB:20% RF:50% LSTM:20%
⚠️ Hata (DOGE): name 'selected_features' is not defined

🔬 ETH ANALİZ EDİLİYOR...
----------------------------------------------------------------------------------------------------
      🔬 FEATURE GATEKEEPER: ETH analiz ediliyor...
         ⚠️ Düşük MI: RSI, Dist_Low, Fear_Index, Weak_